In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import pickle


In [ ]:
data_name_list = ["MERFISH_25", "MERFISH_26", "MERFISH_27", "MERFISH_28", "MERFISH_29"]
results_dict = {}


In [ ]:
# gene set from get_gene_list.R
hvgs_ct = ['Oxt', 'Myh11', 'Pdgfra', 'Penk', 'Scg2', 'Syt2', 'Npas1', 'Trh', 'Calcr', 'Rgs5', 'Selplg', 'Ebf3', 'Rxfp1', 'Crhbp', 'Sema4d', 'Man1a', 'Ttn', 'Gad1', 'Mki67', 'Fn1', 'Ermn', 'Gbx2', 'Th', 'Vgf']
hvgs_clusters = ['Oxt', 'Myh11', 'Selplg', 'Scg2', 'Pdgfra', 'Syt2', 'Trh', 'Npas1', 'Fn1', 'Arhgap36', 'Rgs5', 'Vgf', 'Sst', 'Ebf3', 'Slc17a7', 'Calcr', 'Sema4d', 'Ttn', 'Man1a', 'Gnrh1', 'Crhbp', 'Col25a1', 'Gad1', 'Mbp', 'Ntsr1', 'Mki67', 'Coch', 'Th']
marker_gene_ct = ['Slco1a4', 'Fn1', 'Rgs5','Klf4','Ace2','Gad1','Slc18a2','Gal','Coch','Penk','Aldh1l1','Aqp4','Cxcl14','Pou3f2','Mlc1','Oxt','Cbln1','Ebf3','Slc17a6','Trh','Ermn','Mbp','Sgk1','Opalin','Gjc3','Pdgfra','Traf4','Sox8','Sox6','Selplg','Man1a','Slc15a3','Rgs2','Myh11','Lmod1','Ccnd2','Nnat','Cd24a','Cckbr','Cyr61']
marker_gene_clusters = ['Fn1', 'Slco1a4', 'Rgs5', 'Myh11', 'Klf4', 'Sst', 'Scgn', 'Cyp19a1', 'Crh', 'Vgf', 'Aldh1l1', 'Aqp4', 'Cxcl14', 'Pou3f2', 'Mlc1', 'Oxt', 'Ebf3', 'Trh', 'Cbln1', 'Adcyap1', 'Gal', 'Slc18a2', 'Nts', 'Calcr', 'Scg2', 'Ermn', 'Mbp', 'Sgk1', 'Opalin', 'Gjc3', 'Pdgfra', 'Traf4', 'Sox8', 'Sox6', 'Necab1', 'Ntng1', 'Slc17a6', 'Ramp3', 'Sp9', 'Selplg', 'Man1a', 'Rgs2', 'Slc15a3', 'Nnat', 'Cd24a', 'Cckbr', 'Cyr61']
niche_genes = ['Myh11', 'Syt2', 'Tac2', 'Lmod1', 'Serpinb1b', 'Nts', 'Calcr', 'Esr1', 'Brs3', 'Coch', 'Scgn', 'Sp9', 'Sst', 'Cyp19a1', 'Vgf', 'Mbp', 'Ebf3', 'Ermn', 'Sgk1', 'Opalin', 'Oxt', 'Th', 'Trh', 'Gal', 'Cbln1', 'Necab1', 'Ntng1', 'Slc17a6', 'Ramp3', 'Nnat', 'Cd24a', 'Cckbr', 'Cyr61', 'Mlc1', 'Slc18a2', 'Scg2', 'Egr2', 'Slc17a8']


# calculate kmeans with different r

In [ ]:
my_genes = marker_gene_clusters
for data_name in data_name_list:
    config = load_config("configs/config_cluster_merfish.yaml")
    config.data_name = data_name
    config.refresh_paths()

    name_truth = config.name_truth

    # --- Load data ---
    data_path = str(dataset_file("merfish", config.data_name))
    adata = sc.read_h5ad(data_path) 
    # rename the column of cell_class to cell_type
    adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

    # clean the cell ID to save token
    adata.obs_names = list(range(len(adata)))
    adata.obs_names = adata.obs_names.astype(str)

    # Remove rows with NaN values in 'layer_guess'
    adata = adata[~adata.obs[name_truth].isna()].copy()

    # Verify that NaNs have been removed
    remaining_nan_count = adata.obs[name_truth].isna().sum()
    print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

    pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
    adata.obs = adata.obs.join(pos_data)

    # rename cell types
    celltype_rename = {
        'Astrocyte' : 'Astrocyte',
    'Endothelial 1': 'Endothelial',
    'OD Mature 2': 'Mature oligodendrocytes',
    'Inhibitory': 'Inhibitory',
    'OD Immature 1': 'Immature oligodendrocytes',
    'Excitatory': 'Excitatory',
    'Endothelial 3': 'Endothelial',
    'Microglia': 'Microglia',
    'OD Mature 1': 'Mature oligodendrocytes',
    'Pericytes': 'Pericytes',
    'OD Mature 4': 'Mature oligodendrocytes',
    'Endothelial 2': 'Endothelial',
    'OD Mature 3': 'Mature oligodendrocytes',
    'OD Immature 2': 'Immature oligodendrocytes',
    'Ependymal': 'Ependymal'
    }
    adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
    celltype_data = adata.obs[['cell_type']]

    sc.pp.filter_genes(adata, min_cells=5)
    sc.pp.normalize_total(adata, inplace=True)
    sc.pp.log1p(adata)
    sc.pp.scale(adata)

    # ---- use highly variable genes for each dataset----
    # # Initialize a dictionary to store the top genes per cell type
    # top_genes_per_cell_type = {}

    # for cell_type in adata.obs['cell_type'].unique():
    #     # Subset the data for the current cell type
    #     adata_subset = adata[adata.obs['cell_type'] == cell_type].copy()
    #     if len(adata_subset) < 30:
    #         continue

        
    #     # Compute highly variable genes within the subset
    #     sc.pp.highly_variable_genes(
    #         adata_subset,
    #         n_top_genes=5,
    #         flavor='seurat',
    #         subset=False,
    #         layer=None,
    #         inplace=True
    #     )
        
    #     # Retrieve the top 5 highly variable genes
    #     top_genes = adata_subset.var.loc[adata_subset.var['highly_variable'], :].index.tolist()
        
    #     # Store the results in the dictionary
    #     top_genes_per_cell_type[cell_type] = top_genes

    # # Convert the dictionary to a DataFrame for better visualization
    # top_genes_df = pd.DataFrame.from_dict(top_genes_per_cell_type, orient='index').transpose()

    # # get all the top genes
    # top_genes = list(set(top_genes_df.values.flatten()))

    # ---- use the genes from get_gene_list.R ----
    # common genes between my_genes and var_names
    top_genes = [gene for gene in my_genes if gene in adata.var_names]

    kmeansBoth_ari = []
    kmeans_ari = []
    kemans_genes_ari = []
    kmeansBoth_nmi = []
    kmeans_nmi = []
    kemans_genes_nmi = []

    for r in range(10, 1000, 10):
        adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
        # add diagonal to the adj_matrix
        adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

        # --- Generate one-hot encoded matrix ---
        one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
        one_hot_matrix = one_hot_df.values
        one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

        # --- Calculate neighbor counts ---
        neighbor_count = adj_matrix.dot(one_hot_matrix)
        n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
        # Convert n_neighbors to a column vector for element-wise division
        n_neighbors_col = n_neighbors.reshape(-1, 1)
        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized = neighbor_count / n_neighbors_col

        neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                                    index=celltype_data.index, 
                                    columns=one_hot_df.columns)
        # --- Calculate neighbor genes ---
        neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

        # Perform element-wise division between neighbor_count and n_neighbors_col
        neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

        neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                                    index=adata.obs_names, 
                                    columns=top_genes)
        
        neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()

        # kmeans with both neighbor count and neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_scaled_df)
        kmeansBoth_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeansBoth_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        # kmeans with neighbor count
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df)
        kmeans_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kmeans_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        # kmeans with neighbor genes
        km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
        clusters = km.fit_predict(neighbor_normalized_df_genes)
        kemans_genes_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
        kemans_genes_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
    results_dict[data_name] = {"kmeans_ari": kmeans_ari, "kmeans_genes_ari": kemans_genes_ari, "kmeansBoth_ari": kmeansBoth_ari, "kmeansBoth_nmi": kmeansBoth_nmi, "kmeans_nmi": kmeans_nmi, "kemans_genes_nmi": kemans_genes_nmi}


In [ ]:
# save the results_dict
with open('examples/results/kmeans_results/kmeans_r_merfish_results_dict.pkl', 'wb') as f:
    pickle.dump(results_dict, f)


In [ ]:
with open('examples/results/kmeans_results/kmeans_r_merfish_results_dict.pkl', 'rb') as f:
    results_dict = pickle.load(f)

# plot the trend of ari and nmi with different r

In [ ]:
# plot results_dict in 5 subplots 
fig, axs = plt.subplots(5, 1, figsize=(10, 15))
for i, data_name in enumerate(data_name_list):
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kmeansBoth_ari"], label="kmeansBoth")
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kmeans_ari"], label="kmeans")
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kmeans_genes_ari"], label="kmeans_genes")
    axs[i].legend()
    axs[i].set_title(data_name)
plt.show()

In [ ]:
# plot nmi
fig, axs = plt.subplots(5, 1, figsize=(10, 15))
for i, data_name in enumerate(data_name_list):
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kmeansBoth_nmi"], label="kmeansBoth")
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kmeans_nmi"], label="kmeans")
    axs[i].plot(range(10, 1000, 10), results_dict[data_name]["kemans_genes_nmi"], label="kmeans_genes")
    axs[i].legend()
    axs[i].set_title(data_name)
plt.show()

# plot the boxplot of ari and nmi on certain r

In [ ]:
# Collect ARI scores at r=100
r100_index = range(10, 1000, 10).index(100)  # Find index corresponding to r=100
kmeansBoth_r100 = []
kmeans_r100 = []
kmeans_genes_r100 = []

for data_name in data_name_list:
    kmeansBoth_r100.append(results_dict[data_name]["kmeansBoth_ari"][r100_index])
    kmeans_r100.append(results_dict[data_name]["kmeans_ari"][r100_index])
    kmeans_genes_r100.append(results_dict[data_name]["kmeans_genes_ari"][r100_index])

# Create boxplot
plt.figure(figsize=(8, 6))
box_data = [kmeansBoth_r100, kmeans_r100, kmeans_genes_r100]
plt.boxplot(box_data, labels=['KMeans Both', 'KMeans', 'KMeans Genes'])
plt.title('ARI Scores at r=100 Across Datasets')
plt.ylabel('ARI Score')
plt.grid(True, alpha=0.3)

# Add individual points for each dataset
for i, data in enumerate(box_data, 1):
    plt.scatter([i] * len(data), data, alpha=0.6, color='red', 
                marker='o', s=50, zorder=3)

plt.show()

In [ ]:
# Collect NMI scores at r=100
r100_index = range(10, 1000, 10).index(100)  # Find index corresponding to r=100
kmeansBoth_r100 = []
kmeans_r100 = []
kmeans_genes_r100 = []

for data_name in data_name_list:
	kmeansBoth_r100.append(results_dict[data_name]["kmeansBoth_nmi"][r100_index])
	kmeans_r100.append(results_dict[data_name]["kmeans_nmi"][r100_index])
	kmeans_genes_r100.append(results_dict[data_name]["kemans_genes_nmi"][r100_index])


# Create boxplot
plt.figure(figsize=(8, 6))
box_data = [kmeansBoth_r100, kmeans_r100, kmeans_genes_r100]
plt.boxplot(box_data, labels=['KMeans Both', 'KMeans', 'KMeans Genes'])
plt.title('NMI Scores at r=100 Across Datasets')
plt.ylabel('NMI Score')
plt.grid(True, alpha=0.3)

# Add individual points for each dataset
for i, data in enumerate(box_data, 1):
    plt.scatter([i] * len(data), data, alpha=0.6, color='red', 
                marker='o', s=50, zorder=3)

plt.show()

In [ ]:
kmeansBoth_r100

In [ ]:
# plot difference between nmi and ari
fig, axs = plt.subplots(5, 1, figsize=(10, 15))
for i, data_name in enumerate(data_name_list):
    # Convert lists to numpy arrays for element-wise subtraction
    kmeansBoth_diff = np.array(results_dict[data_name]["kmeansBoth_nmi"]) - np.array(results_dict[data_name]["kmeansBoth_ari"])
    kmeans_diff = np.array(results_dict[data_name]["kmeans_nmi"]) - np.array(results_dict[data_name]["kmeans_ari"]) 
    kmeans_genes_diff = np.array(results_dict[data_name]["kemans_genes_nmi"]) - np.array(results_dict[data_name]["kmeans_genes_ari"])
    
    # Plot the differences
    axs[i].plot(range(10, 1000, 10), kmeansBoth_diff, label="kmeansBoth")
    axs[i].plot(range(10, 1000, 10), kmeans_diff, label="kmeans")
    axs[i].plot(range(10, 1000, 10), kmeans_genes_diff, label="kmeans_genes")
    axs[0].legend()
    axs[i].set_title(data_name)
    axs[i].set_ylabel("NMI - ARI")

plt.show()

# try different gene sets

In [ ]:
r_interest = 100
box_data = []
box_labels = []
gene_names_list = {"hvgs_ct": hvgs_ct, "hvgs_clusters": hvgs_clusters, "marker_gene_ct": marker_gene_ct, "marker_gene_clusters": marker_gene_clusters, "niche": niche_genes}
for gene_type, gene_names in gene_names_list.items():
    my_genes = gene_names
    for data_name in data_name_list:
        config = load_config("configs/config_cluster_merfish.yaml")
        config.data_name = data_name
        config.refresh_paths()

        name_truth = config.name_truth

        # --- Load data ---
        data_path = str(dataset_file("merfish", config.data_name))
        adata = sc.read_h5ad(data_path) 
        # rename the column of cell_class to cell_type
        adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

        # clean the cell ID to save token
        adata.obs_names = list(range(len(adata)))
        adata.obs_names = adata.obs_names.astype(str)

        # Remove rows with NaN values in 'layer_guess'
        adata = adata[~adata.obs[name_truth].isna()].copy()

        # Verify that NaNs have been removed
        remaining_nan_count = adata.obs[name_truth].isna().sum()
        print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

        pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
        adata.obs = adata.obs.join(pos_data)

        # rename cell types
        celltype_rename = {
            'Astrocyte' : 'Astrocyte',
        'Endothelial 1': 'Endothelial',
        'OD Mature 2': 'Mature oligodendrocytes',
        'Inhibitory': 'Inhibitory',
        'OD Immature 1': 'Immature oligodendrocytes',
        'Excitatory': 'Excitatory',
        'Endothelial 3': 'Endothelial',
        'Microglia': 'Microglia',
        'OD Mature 1': 'Mature oligodendrocytes',
        'Pericytes': 'Pericytes',
        'OD Mature 4': 'Mature oligodendrocytes',
        'Endothelial 2': 'Endothelial',
        'OD Mature 3': 'Mature oligodendrocytes',
        'OD Immature 2': 'Immature oligodendrocytes',
        'Ependymal': 'Ependymal'
        }
        adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
        celltype_data = adata.obs[['cell_type']]

        sc.pp.filter_genes(adata, min_cells=5)
        sc.pp.normalize_total(adata, inplace=True)
        sc.pp.log1p(adata)
        sc.pp.scale(adata)

        # ---- use highly variable genes for each dataset----
        # # Initialize a dictionary to store the top genes per cell type
        # top_genes_per_cell_type = {}

        # for cell_type in adata.obs['cell_type'].unique():
        #     # Subset the data for the current cell type
        #     adata_subset = adata[adata.obs['cell_type'] == cell_type].copy()
        #     if len(adata_subset) < 30:
        #         continue

            
        #     # Compute highly variable genes within the subset
        #     sc.pp.highly_variable_genes(
        #         adata_subset,
        #         n_top_genes=5,
        #         flavor='seurat',
        #         subset=False,
        #         layer=None,
        #         inplace=True
        #     )
            
        #     # Retrieve the top 5 highly variable genes
        #     top_genes = adata_subset.var.loc[adata_subset.var['highly_variable'], :].index.tolist()
            
        #     # Store the results in the dictionary
        #     top_genes_per_cell_type[cell_type] = top_genes

        # # Convert the dictionary to a DataFrame for better visualization
        # top_genes_df = pd.DataFrame.from_dict(top_genes_per_cell_type, orient='index').transpose()

        # # get all the top genes
        # top_genes = list(set(top_genes_df.values.flatten()))

        # ---- use the genes from get_gene_list.R ----
        # common genes between my_genes and var_names
        top_genes = [gene for gene in my_genes if gene in adata.var_names]

        kmeansBoth_ari = []
        kmeans_ari = []
        kemans_genes_ari = []
        kmeansBoth_nmi = []
        kmeans_nmi = []
        kemans_genes_nmi = []

        for r in range(90, 120, 10):
            adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
            # add diagonal to the adj_matrix
            adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

            # --- Generate one-hot encoded matrix ---
            one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
            one_hot_matrix = one_hot_df.values
            one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

            # --- Calculate neighbor counts ---
            neighbor_count = adj_matrix.dot(one_hot_matrix)
            n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
            # Convert n_neighbors to a column vector for element-wise division
            n_neighbors_col = n_neighbors.reshape(-1, 1)
            # Perform element-wise division between neighbor_count and n_neighbors_col
            neighbor_matrix_normalized = neighbor_count / n_neighbors_col

            neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                                        index=celltype_data.index, 
                                        columns=one_hot_df.columns)
            # --- Calculate neighbor genes ---
            neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

            # Perform element-wise division between neighbor_count and n_neighbors_col
            neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

            neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                                        index=adata.obs_names, 
                                        columns=top_genes)
            
            neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()

            # kmeans with both neighbor count and neighbor genes
            km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
            clusters = km.fit_predict(neighbor_scaled_df)
            kmeansBoth_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
            kmeansBoth_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
            # kmeans with neighbor count
            km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
            clusters = km.fit_predict(neighbor_normalized_df)
            kmeans_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
            kmeans_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
            # kmeans with neighbor genes
            km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
            clusters = km.fit_predict(neighbor_normalized_df_genes)
            kemans_genes_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))
            kemans_genes_nmi.append(normalized_mutual_info_score(adata.obs[config.name_truth], clusters))
        results_dict[data_name] = {"kmeans_ari": kmeans_ari, "kmeans_genes_ari": kemans_genes_ari, "kmeansBoth_ari": kmeansBoth_ari, "kmeansBoth_nmi": kmeansBoth_nmi, "kmeans_nmi": kmeans_nmi, "kemans_genes_nmi": kemans_genes_nmi}
    # Collect NMI scores at r=100
    r100_index = range(90, 120, 10).index(r_interest)  # Find index corresponding to r=100
    kmeansBoth_r100 = []
    kmeans_r100 = []
    kmeans_genes_r100 = []

    for data_name in data_name_list:
        kmeansBoth_r100.append(results_dict[data_name]["kmeansBoth_nmi"][r100_index])
        kmeans_r100.append(results_dict[data_name]["kmeans_nmi"][r100_index])
        kmeans_genes_r100.append(results_dict[data_name]["kemans_genes_nmi"][r100_index])
    box_data.append(kmeansBoth_r100)
    box_labels.append(gene_type)


In [ ]:
plt.figure(figsize=(8, 8))
plt.boxplot(box_data, labels=box_labels)
plt.title('KMeans NMI Scores with different gene sets')
plt.ylabel('NMI Score')
plt.title(f'NMI Scores at r={r_interest} Across Datasets')
plt.ylabel('NMI Score')
plt.grid(True, alpha=0.3)

# Add individual points for each dataset
for i, data in enumerate(box_data, 1):
    plt.scatter([i] * len(data), data, alpha=0.6, color='red', 
                marker='o', s=50, zorder=3)

plt.show()

In [ ]:
box_data[3]